In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Point Python to the parent directory so it can find the 'src' folder
sys.path.append(os.path.abspath(".."))

from src.model import Problem, State, Stop

# 3. Modeling

In this section, we define the core domain entities (`Problem`, `Request`, `Vehicle`, and `State`) to represent the routing environment. According to the project requirements, the system must enforce a strict `producer > warehouse > delivery_location` flow, meaning goods must pass through the collection center before reaching the customer.

To implement this dynamically:
* We built a `Stop` class to explicitly track actions (`pickup`, `unload`, `load`, `deliver`).
* We implemented `calculate_cost()` to compute the total travel distance using the provided distance matrix.
* We created a modular `is_feasible()` checker capable of evaluating spatial flow and capacity limits progressively.

### Iteration 1: Spatial Flow and Order Fulfillment
For the first iteration, we ignore time windows and vehicle capacity constraints to purely test the spatial routing flow. The test below attempts to load all 40 order items into a single vehicle to verify that the cost calculation and the warehouse flow rules function correctly.

In [ ]:
from src.model import Problem, State, Stop

prob = Problem.from_csv("../data_tiny")

# Group all 40 items by pickup_loc and delivery_loc
by_prod = {}
by_deliv = {}
for r in prob.requests:
    by_prod.setdefault(r.pickup_loc, []).append(r.item_id)
    by_deliv.setdefault(r.delivery_loc, []).append(r.item_id)

all_items = tuple(r.item_id for r in prob.requests)

# Build a dummy Iteration 1 route: Vehicle 1 collects everything, unloads/loads at warehouse, delivers everything
route_v1 = (
    [Stop(loc, "pickup", tuple(items)) for loc, items in by_prod.items()]
    + [Stop(prob.depot_loc, "unload", all_items), Stop(prob.depot_loc, "load", all_items)]
    + [Stop(loc, "deliver", tuple(items)) for loc, items in by_deliv.items()]
)

dummy_state = State(routes={1: route_v1, 2: []})

print("Cost (km):", round(prob.calculate_cost(dummy_state), 2))
print("Feasible (Iteration 1 - No Capacity):", prob.is_feasible(dummy_state, check_capacity=False))
print("Feasible (Iteration 2 - With Capacity):", prob.is_feasible(dummy_state, check_capacity=True))

### Iteration 2: Vehicle Capacity Constraints
Since putting all items into Vehicle 1 exceeds its 150 kg capacity, `is_feasible` correctly returned `False` when checking capacity. In this iteration, we distribute the workload greedily between Vehicle 1 (150 kg limit) and Vehicle 2 (200 kg limit) to prove the model supports multi-vehicle routing without exceeding payload capacities.

In [ ]:
v1_items, v2_items = [], []
v1_weight = 0.0

# Distribute items based on Vehicle 1's 150kg limit
for r in prob.requests:
    if v1_weight + r.weight_kg <= 150.0:
        v1_items.append(r)
        v1_weight += r.weight_kg
    else:
        v2_items.append(r)

def build_simple_route(items):
    if not items:
        return []
    
    by_prod, by_deliv = {}, {}
    for r in items:
        by_prod.setdefault(r.pickup_loc, []).append(r.item_id)
        by_deliv.setdefault(r.delivery_loc, []).append(r.item_id)
        
    all_ids = tuple(r.item_id for r in items)
    
    return (
        [Stop(loc, "pickup", tuple(ids)) for loc, ids in by_prod.items()]
        + [Stop(prob.depot_loc, "unload", all_ids), Stop(prob.depot_loc, "load", all_ids)]
        + [Stop(loc, "deliver", tuple(ids)) for loc, ids in by_deliv.items()]
    )

state_multi = State({
    1: build_simple_route(v1_items),
    2: build_simple_route(v2_items)
})

print("Vehicle 1 Load (kg):", round(v1_weight, 2))
print("Vehicle 2 Load (kg):", round(sum(r.weight_kg for r in v2_items), 2))
print("New Multi-Vehicle Cost (km):", round(prob.calculate_cost(state_multi), 2))
print("Feasible (Iteration 2 - With Capacity):", prob.is_feasible(state_multi, check_capacity=True))